# Scientific Article Search Prototype (OpenAlex + Gemini)

This notebook implements a **simple, explainable prototype** for finding scientific articles that may help estimate an effect for a given **predictor**, **objective**, **context**, and **target group**.

## Pipeline

1. Build several short OpenAlex queries from the input fields
2. Retrieve and deduplicate articles from OpenAlex
3. Apply a lightweight, explainable lexical ranking
4. Optionally rerank a small shortlist with Gemini
5. Display both the retrieval diagnostics and the final ranked shortlist

## Design choices

- **OpenAlex** is used for recall
- **Heuristic ranking** is intentionally simple and interpretable
- **Gemini** is optional and only used on a small shortlist
- The notebook remains usable even when Gemini quota is unavailable

In [2]:
import os
import re
import json
import time
import math
import requests
import pandas as pd
from IPython.display import display
from datetime import datetime
from typing import List, Dict, Any, Optional

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)

## 1. Secrets and configuration

This notebook expects a Kaggle secret called `GEMINI_API_KEY`.

If Gemini quota is unavailable, the notebook will still run in **heuristic-only mode**.

In [3]:
import os

GEMINI_API_KEY = None


try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", None)

GEMINI_AVAILABLE = bool(GEMINI_API_KEY)
print("Gemini key available:", GEMINI_AVAILABLE)


Gemini key available: True


## 2. Search inputs

Edit these values to test a different use case.

In [34]:
#  INPUT 
objective = "socio emotional development"
context = "parenting support program"
group = "children"
predictor = "family support"
tags = "early childhood, parenting"

#  PARAMETERS 
YEAR_MIN = 2000
OPENALEX_PER_PAGE = 50
OPENALEX_MAX_PAGES_PER_QUERY = 2

MAX_CANDIDATES_AFTER_DEDUP = 250


TOP_N_FOR_LLM = 24   # réduit pour économiser le quota
GEMINI_BATCH_SIZE =   8 # max 8 articles par appel


OPENALEX_MAILTO = "akram.chouichi.etu@univ-lille.fr"  

## 3. Text helpers

These helpers normalize text, build term lists, and create lightweight lexical signals.

In [35]:
def normalize_text(s: Any) -> str:
    if s is None:
        return ""
    try:
        if pd.isna(s):
            return ""
    except Exception:
        pass
    s = str(s).lower().strip()
    s = re.sub(r"[/_\\-]+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


STOPWORDS = {
    "the", "a", "an", "of", "to", "for", "and", "or", "in", "on", "at", "by", "with",
    "from", "about", "into", "over", "after", "before", "between", "through", "during",
    "without", "under", "within", "is", "are", "was", "were", "be", "been", "being",
    "this", "that", "these", "those", "it", "its", "their", "his", "her",
    "program", "programme", "intervention", "study", "trial"
}


def tokenize(text: Any) -> List[str]:
    text = normalize_text(text)
    toks = text.split()
    return [t for t in toks if len(t) >= 3 and t not in STOPWORDS]


def unique_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def field_terms(s: str, max_terms: int = 6) -> List[str]:
    return unique_preserve_order(tokenize(s))[:max_terms]

## 4. Generic group expansion and effect-size lexical signal

This is **not** a domain-specific predictor dictionary.  
It only lightly expands common population labels and looks for generic quantitative/effect-study language.

In [36]:
GROUP_EXPANSIONS = {
    "children": ["child", "children", "youth", "adolescent", "adolescents", "pediatric", "school-age"],
    "parents": ["parent", "parents", "caregiver", "caregivers", "mother", "father", "families"],
    "adults": ["adult", "adults"],
    "elderly": ["elderly", "older adults", "seniors", "aging"],
    "students": ["student", "students", "pupil", "pupils"],
    "teachers": ["teacher", "teachers", "educator", "educators"],
}


def expand_group_terms(group: str) -> List[str]:
    g = normalize_text(group)
    if not g:
        return []
    if g in GROUP_EXPANSIONS:
        return GROUP_EXPANSIONS[g]
    base = field_terms(g, max_terms=6)
    return unique_preserve_order([g] + base)


EFFECT_SIZE_TERMS = [
    "effect size",
    "cohen",
    "cohen s d",
    "cohens d",
    "hedges g",
    "hedge s g",
    "standardized mean difference",
    "standardised mean difference",
    "smd",
    "odds ratio",
    "risk ratio",
    "relative risk",
    "meta analysis",
    "systematic review",
    "randomized trial",
    "randomised trial",
    "controlled trial",
    "intervention study",
    "pre post",
    "regression coefficient",
]


def effect_size_signal(text: str) -> float:
    t = normalize_text(text)
    matches = sum(1 for term in EFFECT_SIZE_TERMS if term in t)
    if matches == 0:
        return 0.0
    if matches == 1:
        return 0.4
    if matches == 2:
        return 0.7
    return 1.0

## 5. Build OpenAlex queries

The goal is to improve recall with several short, generic queries instead of relying on one exact query.

In [37]:
def reconstruct_abstract(inv_index: Optional[dict]) -> str:
    if not inv_index:
        return ""
    words = []
    for word, positions in inv_index.items():
        for pos in positions:
            words.append((pos, word))
    words.sort(key=lambda x: x[0])
    return " ".join(word for _, word in words)

def build_openalex_queries(
    objective: str,
    context: str,
    group: str,
    predictor: str,
    tags: str
) -> List[str]:
    obj_terms = field_terms(objective, max_terms=5)
    ctx_terms = field_terms(context, max_terms=5)
    pred_terms = field_terms(predictor, max_terms=5)
    group_terms = expand_group_terms(group)[:3]
    tag_terms = field_terms(tags, max_terms=4)

    queries = []

    if pred_terms and ctx_terms:
        queries.append(" ".join(pred_terms[:3] + ctx_terms[:3]))

    if pred_terms and obj_terms:
        queries.append(" ".join(pred_terms[:3] + obj_terms[:3]))

    if obj_terms and ctx_terms:
        queries.append(" ".join(obj_terms[:3] + ctx_terms[:3] + group_terms[:2]))

    if pred_terms and group_terms:
        queries.append(" ".join(pred_terms[:3] + group_terms[:2] + tag_terms[:2]))

    if tag_terms and obj_terms:
        queries.append(" ".join(tag_terms[:3] + obj_terms[:3]))

    all_terms = unique_preserve_order(
        pred_terms[:3] + ctx_terms[:3] + obj_terms[:3] + group_terms[:2] + tag_terms[:2]
    )
    if all_terms:
        queries.append(" ".join(all_terms[:10]))

    return unique_preserve_order([q.strip() for q in queries if q.strip()])

## 6. OpenAlex retrieval and deduplication

In [38]:
OPENALEX_API = "https://api.openalex.org/works"


def openalex_search(
    query: str,
    year_min: int,
    per_page: int = 50,
    max_pages: int = 2,
    mailto: Optional[str] = None
) -> pd.DataFrame:
    results = []
    cursor = "*"
    headers = {"User-Agent": "elementsimpact-case-prototype/0.3"}

    for _ in range(max_pages):
        params = {
            "search": query,
            "filter": f"type:article|review,from_publication_date:{year_min}-01-01",
            "per-page": per_page,
            "cursor": cursor,
            "sort": "relevance_score:desc"
        }
        if mailto:
            params["mailto"] = mailto

        try:
            resp = requests.get(OPENALEX_API, params=params, headers=headers, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"[OpenAlex ERROR] query='{query}' -> {e}")
            break

        works = data.get("results", [])
        if not works:
            break

        for w in works:
            primary_location = w.get("primary_location") or {}
            source = primary_location.get("source") or {}

            results.append({
                "id": w.get("id"),
                "doi": w.get("doi"),
                "title": w.get("title", ""),
                "year": w.get("publication_year"),
                "journal": source.get("display_name", ""),
                "landing_page_url": primary_location.get("landing_page_url") or primary_location.get("pdf_url"),
                "cited_by_count": w.get("cited_by_count", 0),
                "abstract": reconstruct_abstract(w.get("abstract_inverted_index")),
                "query_used": query,
            })

        next_cursor = (data.get("meta") or {}).get("next_cursor")
        if not next_cursor:
            break
        cursor = next_cursor
        time.sleep(0.2)

    df = pd.DataFrame(results)
    if df.empty:
        return df

    for col in ["title", "journal", "abstract", "landing_page_url", "query_used", "id", "doi"]:
        if col not in df.columns:
            df[col] = ""

    return df


def retrieve_candidates(
    objective: str,
    context: str,
    group: str,
    predictor: str,
    tags: str,
    year_min: int,
    per_page: int,
    max_pages: int,
    mailto: Optional[str] = None
) -> pd.DataFrame:
    queries = build_openalex_queries(
        objective=objective,
        context=context,
        group=group,
        predictor=predictor,
        tags=tags
    )

    print("Queries sent to OpenAlex:")
    for i, q in enumerate(queries, 1):
        print(f"  {i}. {q}")

    frames = []
    for q in queries:
        df_q = openalex_search(
            query=q,
            year_min=year_min,
            per_page=per_page,
            max_pages=max_pages,
            mailto=mailto
        )
        if not df_q.empty:
            frames.append(df_q)

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)

    df["dedup_key"] = df["doi"].fillna("").replace("", pd.NA)
    df["dedup_key"] = df["dedup_key"].fillna(df["id"])
    df["dedup_key"] = df["dedup_key"].fillna(
        df["title"].fillna("").str.lower().str.strip() + "|" + df["year"].astype(str)
    )

    hit_count = df.groupby("dedup_key").size().rename("query_hit_count").reset_index()
    df = df.merge(hit_count, on="dedup_key", how="left")

    df = (
        df.sort_values(by=["query_hit_count", "cited_by_count", "year"], ascending=[False, False, False])
          .drop_duplicates(subset=["dedup_key"], keep="first")
          .reset_index(drop=True)
    )

    return df

## 7. Explainable heuristic ranking

The heuristic score is deliberately lightweight.  
It helps prioritize a shortlist for manual review or optional Gemini reranking.

In [39]:
def overlap_score(text: str, terms: List[str]) -> float:
    if not terms:
        return 0.0
    t = normalize_text(text)
    hits = sum(1 for term in terms if normalize_text(term) in t)
    return hits / len(terms)


def exact_phrase_bonus(text: str, phrase: str) -> float:
    p = normalize_text(phrase)
    if not p:
        return 0.0
    t = normalize_text(text)
    return 1.0 if p in t else 0.0


def metadata_quality_score(abstract: str, year: Any, cited_by_count: Any) -> float:
    score = 0.0

    if normalize_text(abstract):
        score += 0.4

    try:
        y = int(year)
        age = max(0, datetime.now().year - y)
        score += max(0.0, 0.4 - min(age, 20) * 0.02)
    except Exception:
        pass

    try:
        c = int(cited_by_count)
        score += min(math.log1p(c) / 6, 0.2)
    except Exception:
        pass

    return min(score, 1.0)


def compute_heuristic_scores(
    df: pd.DataFrame,
    objective: str,
    context: str,
    group: str,
    predictor: str,
    tags: str
) -> pd.DataFrame:
    out = df.copy()

    predictor_terms = field_terms(predictor, max_terms=5)
    context_terms = field_terms(context, max_terms=5)
    objective_terms = field_terms(objective, max_terms=5)
    tag_terms = field_terms(tags, max_terms=4)
    group_terms = expand_group_terms(group)[:5]

    pred_phrase = normalize_text(predictor)
    ctx_phrase = normalize_text(context)
    obj_phrase = normalize_text(objective)

    rows = []
    for _, row in out.iterrows():
        title = row.get("title", "")
        abstract = row.get("abstract", "")
        text = f"{title} {abstract}".strip()

        full_text = normalize_text(text)
        title_text = normalize_text(title)

        predictor_overlap = overlap_score(full_text, predictor_terms)
        context_overlap = overlap_score(full_text, context_terms)
        objective_overlap = overlap_score(full_text, objective_terms)
        group_overlap = overlap_score(full_text, group_terms)
        tags_overlap = overlap_score(full_text, tag_terms)

        predictor_phrase = exact_phrase_bonus(full_text, pred_phrase)
        context_phrase = exact_phrase_bonus(full_text, ctx_phrase)
        objective_phrase = exact_phrase_bonus(full_text, obj_phrase)

        effect_signal = effect_size_signal(text)
        metadata_signal = metadata_quality_score(
            abstract=abstract,
            year=row.get("year"),
            cited_by_count=row.get("cited_by_count", 0)
        )

        title_pred_bonus = exact_phrase_bonus(title_text, pred_phrase)
        title_ctx_bonus = exact_phrase_bonus(title_text, ctx_phrase)

        score = (
            0.183 * predictor_overlap +
            0.150 * context_overlap +
            0.133 * objective_overlap +
            0.067 * group_overlap +
            0.050 * tags_overlap +
            0.083 * predictor_phrase +
            0.050 * context_phrase +
            0.033 * objective_phrase +
            0.042 * title_pred_bonus +
            0.025 * title_ctx_bonus +
            0.117 * effect_signal +
            0.067 * metadata_signal
        )

        rows.append({
            "predictor_overlap": round(predictor_overlap, 3),
            "context_overlap": round(context_overlap, 3),
            "objective_overlap": round(objective_overlap, 3),
            "group_overlap": round(group_overlap, 3),
            "tags_overlap": round(tags_overlap, 3),
            "predictor_phrase": round(predictor_phrase, 3),
            "context_phrase": round(context_phrase, 3),
            "objective_phrase": round(objective_phrase, 3),
            "effect_signal": round(effect_signal, 3),
            "metadata_signal": round(metadata_signal, 3),
            "heuristic_score": round(score, 4),
        })

    score_df = pd.DataFrame(rows)
    out = pd.concat([out.reset_index(drop=True), score_df], axis=1)

    out = out.sort_values(
        by=["heuristic_score", "query_hit_count", "effect_signal", "cited_by_count", "year"],
        ascending=[False, False, False, False, False]
    ).reset_index(drop=True)

    return out

## 8. Retrieve and rank with heuristics

In [40]:
df_candidates = retrieve_candidates(
    objective=objective,
    context=context,
    group=group,
    predictor=predictor,
    tags=tags,
    year_min=YEAR_MIN,
    per_page=OPENALEX_PER_PAGE,
    max_pages=OPENALEX_MAX_PAGES_PER_QUERY,
    mailto=OPENALEX_MAILTO
)

print("Number of unique candidates:", len(df_candidates))

if df_candidates.empty:
    raise ValueError("No candidates returned by OpenAlex.")

df_candidates = df_candidates.head(MAX_CANDIDATES_AFTER_DEDUP).copy()

df_ranked = compute_heuristic_scores(
    df=df_candidates,
    objective=objective,
    context=context,
    group=group,
    predictor=predictor,
    tags=tags
)

display(df_ranked[[
    "title", "year", "journal", "query_hit_count",
    "heuristic_score", "predictor_overlap", "context_overlap",
    "objective_overlap", "group_overlap", "effect_signal",
    "cited_by_count", "landing_page_url"
]].head(15))

Queries sent to OpenAlex:
  1. family support parenting support
  2. family support socio emotional development
  3. socio emotional development parenting support child children
  4. family support child children early childhood
  5. early childhood parenting socio emotional development
  6. family support parenting socio emotional development child children early childhood
Number of unique candidates: 475


,title,year,journal,query_hit_count,heuristic_score,predictor_overlap,context_overlap,objective_overlap,group_overlap,effect_signal,cited_by_count,landing_page_url
0,Promoting Childhood Development Globally Through Caregiving Interventions,2023,PEDIATRICS,3,0.7826,1.0,1.0,1.000,0.8,1.0,13,https://doi.org/10.1542/peds.2023-060221b
1,A new era in child maltreatment prevention: call to action,2023,The Medical Journal of Australia,2,0.7620,1.0,1.0,1.000,1.0,0.0,14,https://doi.org/10.5694/mja2.51872
2,Psychological care of children and adolescents with diabetes,2009,Pediatric Diabetes,4,0.7356,1.0,0.5,1.000,1.0,1.0,140,https://doi.org/10.1111/j.1399-5448.2009.00580.x
3,Psychological care of children and adolescents with diabetes,2007,Pediatric Diabetes,4,0.6978,1.0,0.5,1.000,1.0,0.7,175,https://doi.org/10.1111/j.1399-5448.2007.00318.x
4,How is participation in parent-child-interaction-focused and parenting-skills-focused courses associated with child ...,2017,Early Years Journal of International Research and Development,2,0.6474,1.0,1.0,1.000,0.4,0.0,13,https://doi.org/10.1080/09575146.2017.1288089
5,Promotion and prevention in child mental health,2009,Indian Journal of Psychiatry,3,0.6272,1.0,1.0,1.000,1.0,0.0,58,https://doi.org/10.4103/0019-5545.49447
6,Emotional Warmth and Rejection Parenting Styles of Grandparents/Great Grandparents and the Social–Emotional Developm...,2023,International Journal of Environmental Research and Public Health,2,0.6221,1.0,1.0,1.000,0.4,0.0,6,https://doi.org/10.3390/ijerph20021568
7,Children of Mentally III Parents at Risk Evaluation (COMPARE): Design and Methods of a Randomized Controlled Multice...,2019,Frontiers in Psychiatry,2,0.6139,1.0,1.0,1.000,0.4,0.4,39,https://doi.org/10.3389/fpsyt.2019.00128
8,Supplement to the JCIH 2007 Position Statement: Principles and Guidelines for Early Intervention After Confirmation ...,2013,PEDIATRICS,1,0.6046,1.0,0.5,0.667,0.4,0.7,241,https://doi.org/10.1542/peds.2013-0008
9,Stimulating Parenting Practices in Indigenous and Non-Indigenous Mexican Communities,2017,International Journal of Environmental Research and Public Health,2,0.5974,1.0,1.0,1.000,0.4,0.0,20,https://doi.org/10.3390/ijerph15010029


## 9. Optional Gemini reranking

Gemini is used only on a small shortlist.  
If the key is missing or quota is exhausted, the notebook falls back to heuristic-only ranking.

In [42]:
pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.


In [43]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_AVAILABLE else None
# 
GEMINI_MODEL = "gemini-2.5-flash"


In [44]:
def build_batch_prompt(
    objective: str,
    context: str,
    group: str,
    predictor: str,
    tags: str,
    articles: List[Dict[str, Any]]
) -> str:
    project_block = f"""
You are a research assistant helping identify scientific articles that could support the estimation of effect sizes.

PROJECT DESCRIPTION
Objective: {objective}
Context: {context}
Target group: {group}
Predictor / intervention of interest: {predictor}
Tags: {tags}

GOAL
We want to identify studies that could help estimate the effect of the predictor on the objective outcome.

IMPORTANT PRINCIPLE
An article can be useful for estimating an effect even if it does NOT explicitly report "effect size".

Examples of usable quantitative evidence:
- randomized controlled trials
- quasi-experiments
- intervention vs control comparisons
- regression models linking predictor and outcome
- odds ratios or risk ratios
- mean differences between groups
- meta-analyses summarizing quantitative results

Articles that are usually NOT usable for effect estimation:
- commentary or opinion pieces
- policy papers
- calls to action
- conceptual discussions without empirical data
- narrative reviews without pooled quantitative results
- protocols without results

TASK
For each article, evaluate:
1. How relevant the article is to the research question
2. Whether the article likely contains quantitative evidence that could allow estimating an effect

RELEVANCE LABELS
- highly_relevant: strong match between predictor, context, and group
- somewhat_relevant: partially related but context or predictor differs
- background_only: related conceptually but not directly useful
- irrelevant: unrelated to the research question

usable_for_effect_estimation = true if the article likely contains:
- quantitative comparison
- empirical data
- regression or statistical association
- intervention evaluation
- meta-analysis with pooled results

usable_for_effect_estimation = false if the article is:
- commentary
- conceptual discussion
- policy paper
- descriptive text without quantitative analysis

Return ONLY valid JSON.
Return a list of objects with this structure:
[
  {{
    "article_index": integer,
    "relevance_label": "highly_relevant | somewhat_relevant | background_only | irrelevant",
    "usable_for_effect_estimation": true or false,
    "study_type": "trial | observational | meta_analysis | review | commentary | unclear",
    "action_match": integer,
    "group_match": integer,
    "quantitative_signal": integer,
    "short_reason": "max 30 words"
  }}
]

SCORING GUIDELINES
- action_match: 0 = predictor absent, 1 = related concept, 2 = direct match
- group_match: 0 = different population, 1 = somewhat related, 2 = same population
- quantitative_signal: 0 = no quantitative evidence, 1 = some quantitative discussion, 2 = clear quantitative empirical study
"""

    article_blocks = []
    for idx, a in enumerate(articles):
        abstract_short = str(a.get("abstract", "") or "")[:300]  # tronqué pour limiter les tokens
        article_blocks.append(f"""
ARTICLE {idx}
title: {a.get('title', '')}
year: {a.get('year', '')}
journal: {a.get('journal', '')}
abstract: {abstract_short}
""")
    return project_block + "\n".join(article_blocks)


def call_gemini_with_retry(prompt: str, max_retries: int = 4, sleep_base: float = 3.0) -> str:
    if not GEMINI_AVAILABLE:
        raise RuntimeError("Gemini key not available.")

    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )
            text = response.text
            return text.strip() if text else ""

        except Exception as e:
            last_error = e
            err_text = str(e)

            if "RESOURCE_EXHAUSTED" in err_text or "429" in err_text:
                m = re.search(r"retry in ([0-9]+(?:\.[0-9]+)?)s", err_text, flags=re.IGNORECASE)
                retry_delay = float(m.group(1)) if m else None

                if "generate_content_free_tier_requests" in err_text:
                    print("[Gemini STOP] Free-tier request quota exhausted for this model/project.")
                    raise RuntimeError(err_text)

                wait_s = retry_delay if retry_delay is not None else sleep_base * (2 ** attempt)
                print(f"[Gemini retry {attempt+1}/{max_retries}] 429 -> sleep {wait_s:.1f}s")
                time.sleep(wait_s)
                continue

            wait_s = sleep_base * (2 ** attempt)
            print(f"[Gemini retry {attempt+1}/{max_retries}] {e} -> sleep {wait_s:.1f}s")
            time.sleep(wait_s)

    raise RuntimeError(f"Gemini failed after retries: {last_error}")


def extract_json_block(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE).strip()
        text = re.sub(r"```$", "", text.strip()).strip()
    return text


def parse_gemini_json(text: str) -> List[Dict[str, Any]]:
    cleaned = extract_json_block(text)
    data = json.loads(cleaned)
    if not isinstance(data, list):
        raise ValueError("Gemini output is not a list.")
    return data

In [45]:
def gemini_score_batches(
    df_ranked: pd.DataFrame,
    objective: str,
    context: str,
    group: str,
    predictor: str,
    tags: str,
    top_n: int = 8,
    batch_size: int = 8
) -> pd.DataFrame:
    subset = df_ranked.head(top_n).copy().reset_index(drop=True)

    if not GEMINI_AVAILABLE:
        subset["gemini_label"] = "not_scored"
        subset["gemini_effect_estimation_possible"] = False
        subset["gemini_study_type"] = "unclear"
        subset["gemini_action_match"] = 0
        subset["gemini_group_match"] = 0
        subset["gemini_quantitative_signal"] = 0
        subset["gemini_reason"] = "Gemini key unavailable."
        return subset

    gemini_rows = []
    gemini_failed_hard = False

    for start in range(0, len(subset), batch_size):
        end = min(start + batch_size, len(subset))
        print(f"Processing Gemini batch rows {start} to {end-1}")

        batch_df = subset.iloc[start:end].copy().reset_index(drop=True)
        articles = batch_df[["title", "abstract", "year", "journal"]].to_dict(orient="records")

        prompt = build_batch_prompt(
            objective=objective,
            context=context,
            group=group,
            predictor=predictor,
            tags=tags,
            articles=articles
        )

        try:
            raw_text = call_gemini_with_retry(prompt)
            parsed = parse_gemini_json(raw_text)
            parsed_by_idx = {int(x["article_index"]): x for x in parsed if "article_index" in x}

            for local_idx in range(len(batch_df)):
                item = parsed_by_idx.get(local_idx, {})
                gemini_rows.append({
                    "global_row_idx": start + local_idx,
                    "gemini_label": item.get("relevance_label", "error"),
                    "gemini_effect_estimation_possible": bool(item.get("usable_for_effect_estimation", False)),
                    "gemini_study_type": item.get("study_type", "unclear"),
                    "gemini_action_match": item.get("action_match", 0),
                    "gemini_group_match": item.get("group_match", 0),
                    "gemini_quantitative_signal": item.get("quantitative_signal", 0),
                    "gemini_reason": item.get("short_reason", "No reason returned.")
                })

        except Exception as e:
            print(f"[Gemini batch ERROR] rows {start}:{end} -> {e}")
            gemini_failed_hard = True
            break

        time.sleep(15.0)  # pause entre batches pour respecter le rate limit

    if gemini_failed_hard:
        subset["gemini_label"] = "not_scored"
        subset["gemini_effect_estimation_possible"] = False
        subset["gemini_study_type"] = "unclear"
        subset["gemini_action_match"] = 0
        subset["gemini_group_match"] = 0
        subset["gemini_quantitative_signal"] = 0
        subset["gemini_reason"] = "Gemini unavailable: quota exhausted or parsing failed."
        return subset

    gemini_df = pd.DataFrame(gemini_rows).sort_values("global_row_idx").reset_index(drop=True)
    subset = pd.concat([subset.reset_index(drop=True), gemini_df.drop(columns=["global_row_idx"])], axis=1)
    return subset


def rerank_final(df_gemini: pd.DataFrame) -> pd.DataFrame:
    out = df_gemini.copy()

    label_priority = {
        "highly_relevant": 4,
        "somewhat_relevant": 3,
        "background_only": 2,
        "irrelevant": 1,
        "not_scored": 0,
        "error": 0
    }

    out["gemini_label_priority"] = out["gemini_label"].map(label_priority).fillna(0)
    out["gemini_effect_priority"] = out["gemini_effect_estimation_possible"].fillna(False).astype(int)

    if "gemini_action_match" not in out.columns:
        out["gemini_action_match"] = 0
    if "gemini_group_match" not in out.columns:
        out["gemini_group_match"] = 0
    if "gemini_quantitative_signal" not in out.columns:
        out["gemini_quantitative_signal"] = 0

    gemini_available_now = ("gemini_label" in out.columns) and (out["gemini_label"] != "not_scored").any()

    if gemini_available_now:
        out["final_score"] = (
            0.45 * out["gemini_label_priority"] +
            0.20 * out["gemini_effect_priority"] +
            0.10 * out["gemini_action_match"].fillna(0) +
            0.08 * out["gemini_group_match"].fillna(0) +
            0.07 * out["gemini_quantitative_signal"].fillna(0) +
            0.10 * out["heuristic_score"].fillna(0)
        )
    else:
        out["final_score"] = out["heuristic_score"]

    out = out.sort_values(
        by=["final_score", "heuristic_score", "query_hit_count", "cited_by_count", "year"],
        ascending=[False, False, False, False, False]
    ).reset_index(drop=True)

    out["rank"] = range(1, len(out) + 1)
    return out

## 10. Run Gemini reranking and inspect diagnostics

In [46]:
df_llm = gemini_score_batches(
    df_ranked=df_ranked,
    objective=objective,
    context=context,
    group=group,
    predictor=predictor,
    tags=tags,
    top_n=TOP_N_FOR_LLM,
    batch_size=GEMINI_BATCH_SIZE
)

print("Candidates after OpenAlex dedup:", len(df_candidates))
print("Candidates after heuristic ranking:", len(df_ranked))
print("Sent to Gemini:", min(TOP_N_FOR_LLM, len(df_ranked)))
print("Rows returned by Gemini dataframe:", len(df_llm))
print("df_llm columns:", df_llm.columns.tolist())

if "gemini_label" in df_llm.columns:
    print("\nGemini label counts:")
    print(df_llm["gemini_label"].value_counts(dropna=False))

if "gemini_reason" in df_llm.columns:
    print("\nRows with Gemini error/not_scored:")
    err = df_llm[df_llm["gemini_label"].isin(["error", "not_scored"])][["title", "gemini_reason"]]
    display(err.head(20))

df_final = rerank_final(df_llm)
print("df_final shape:", df_final.shape)
display(df_final.head(15))

Processing Gemini batch rows 0 to 7
Processing Gemini batch rows 8 to 15
Processing Gemini batch rows 16 to 23
Candidates after OpenAlex dedup: 250
Candidates after heuristic ranking: 250
Sent to Gemini: 24
Rows returned by Gemini dataframe: 24
df_llm columns: ['id', 'doi', 'title', 'year', 'journal', 'landing_page_url', 'cited_by_count', 'abstract', 'query_used', 'dedup_key', 'query_hit_count', 'predictor_overlap', 'context_overlap', 'objective_overlap', 'group_overlap', 'tags_overlap', 'predictor_phrase', 'context_phrase', 'objective_phrase', 'effect_signal', 'metadata_signal', 'heuristic_score', 'gemini_label', 'gemini_effect_estimation_possible', 'gemini_study_type', 'gemini_action_match', 'gemini_group_match', 'gemini_quantitative_signal', 'gemini_reason']

Gemini label counts:
gemini_label
background_only      7
highly_relevant      6
somewhat_relevant    6
irrelevant           5
Name: count, dtype: int64

Rows with Gemini error/not_scored:


,title,gemini_reason


df_final shape: (24, 33)


,id,doi,title,year,journal,landing_page_url,cited_by_count,abstract,query_used,dedup_key,query_hit_count,predictor_overlap,context_overlap,objective_overlap,group_overlap,tags_overlap,predictor_phrase,context_phrase,objective_phrase,effect_signal,metadata_signal,heuristic_score,gemini_label,gemini_effect_estimation_possible,gemini_study_type,gemini_action_match,gemini_group_match,gemini_quantitative_signal,gemini_reason,gemini_label_priority,gemini_effect_priority,final_score,rank
0,https://openalex.org/W2589497849,https://doi.org/10.1080/09575146.2017.1288089,How is participation in parent-child-interaction-focused and parenting-skills-focused courses associated with child ...,2017,Early Years Journal of International Research and Development,https://doi.org/10.1080/09575146.2017.1288089,13,The present study examines the effects of a family support program on children’s socio-emotional and language develo...,family support socio emotional development,https://doi.org/10.1080/09575146.2017.1288089,2,1.0,1.0,1.000,0.4,0.333,1.0,0.0,0.0,0.0,0.82,0.6474,highly_relevant,True,observational,2,2,2,Directly examines effects of a family support program on children's socio-emotional development with quantitative an...,4,1,2.56474,1
1,https://openalex.org/W2778698446,https://doi.org/10.3390/ijerph15010029,Stimulating Parenting Practices in Indigenous and Non-Indigenous Mexican Communities,2017,International Journal of Environmental Research and Public Health,https://doi.org/10.3390/ijerph15010029,20,Parenting may be influenced by ethnicity; marginalization; education; and poverty. A critical but unexamined questio...,family support socio emotional development,https://doi.org/10.3390/ijerph15010029,2,1.0,1.0,1.000,0.4,0.333,0.0,0.0,1.0,0.0,0.82,0.5974,highly_relevant,True,observational,2,2,2,"Examines associations between factors and stimulating parenting practices in Mexican communities, containing quantit...",4,1,2.55974,2
2,https://openalex.org/W4399393584,https://doi.org/10.3390/children11060695,Nurturing Sustainable Development: The Interplay of Parenting Styles and SDGs in Children’s Development,2024,Children,https://doi.org/10.3390/children11060695,19,This study delves into the dynamics of parenting styles and their impact on the cognitive and social-affective devel...,family support socio emotional development,https://doi.org/10.3390/children11060695,2,1.0,1.0,1.000,0.4,0.333,0.0,0.0,0.0,0.0,0.96,0.5738,highly_relevant,True,observational,2,2,2,Investigates the impact of parenting styles on children's cognitive and social-affective development using empirical...,4,1,2.55738,3
3,https://openalex.org/W2138312879,https://doi.org/10.1037/a0013858,Collateral benefits of the family check-up on early childhood school readiness: Indirect effects of parents' positiv...,2008,Developmental Psychology,https://doi.org/10.1037/a0013858,158,The authors examined the longitudinal effects of the Family Check-Up (FCU) on parents' positive behavior support and...,family support parenting support,https://doi.org/10.1037/a0013858,2,1.0,1.0,0.333,0.4,1.000,0.0,0.0,0.0,0.4,0.64,0.5438,highly_relevant,True,observational,2,2,2,Examined longitudinal effects of a family intervention (FCU) on early childhood school readiness.,4,1,2.55438,4
4,https://openalex.org/W2035940131,https://doi.org/10.1111/j.1467-6494.2005.00347.x,A Longitudinal Study of the Relationship of Maternal Autonomy Support to Children's Adjustment and Achievement in Sc...,2005,Journal of Personality,https://doi.org/10.1111/j.1467-6494.2005.00347.x,217,A longitudinal study examined the relations of maternal autonomy support to children's school adjustment. Autonomy s...,family support child children early childhood,https://doi.org/10.1111/j.1467-6494.2005.00347.x,1,1.0,1.0,0.667,0.4,1.000,0.0,0.0,0.0,0.0,0.60,0.5387,highly_relevant,True,observational,2,2,2,Longitudinal study examining relations of maternal autonomy support to children's school and social adjustment.,4,1,2.55387,5
5,https://open

## 11. Display helpers

We keep two views:
- a **retrieval view** for OpenAlex + heuristics
- a **final view** for the Gemini-reranked shortlist

In [50]:
def clean_url(url):
    if pd.isna(url) or not url:
        return ""
    url = str(url)
    return url if len(url) <= 60 else url[:57] + "..."


def wrap_text(s, width=55):
    if pd.isna(s) or s is None:
        return ""
    s = str(s)
    words = s.split()
    lines = []
    line = []
    current = 0
    for w in words:
        if current + len(w) + len(line) <= width:
            line.append(w)
            current += len(w)
        else:
            lines.append(" ".join(line))
            line = [w]
            current = len(w)
    if line:
        lines.append(" ".join(line))
    return "<br>".join(lines)


def color_label(val):
    if val == "highly_relevant":
        return "background-color: #c6efce; color: black;"
    if val == "somewhat_relevant":
        return "background-color: #ffeb9c; color: black;"
    if val == "background_only":
        return "background-color: #ddebf7; color: black;"
    if val == "irrelevant":
        return "background-color: #f4cccc; color: black;"
    if val in ["error", "not_scored"]:
        return "background-color: #d9d9d9; color: black;"
    return ""


def color_boolean(val):
    if val is True:
        return "background-color: #c6efce; color: black;"
    if val is False:
        return "background-color: #f4cccc; color: black;"
    return ""


def build_final_view(df, n=8):
    tmp = df.head(n).copy()

    tmp["Title"] = tmp["title"].apply(lambda x: wrap_text(x, 60))
    tmp["Journal"] = tmp["journal"].apply(lambda x: wrap_text(x, 25))
    tmp["Reason"] = tmp["gemini_reason"].apply(lambda x: wrap_text(x, 55))
    tmp["URL"] = tmp["landing_page_url"].apply(clean_url)

    cols = [
        "rank", "Title", "year", "Journal",
        "heuristic_score", "final_score",
        "gemini_label", "gemini_effect_estimation_possible",
        "gemini_action_match", "gemini_group_match", "gemini_quantitative_signal",
        "Reason", "URL"
    ]

    tmp = tmp[cols].rename(columns={
        "rank": "Rank",
        "year": "Year",
        "heuristic_score": "Heuristic",
        "final_score": "Final",
        "gemini_label": "Gemini label",
        "gemini_effect_estimation_possible": "Effect estimation?",
        "gemini_action_match": "Action",
        "gemini_group_match": "Group",
        "gemini_quantitative_signal": "QuantSignal"
    })

    return (
        tmp.style
        .format({
            "Heuristic": "{:.3f}",
            "Final": "{:.3f}"
        })
        .map(color_label, subset=["Gemini label"])
        .map(color_boolean, subset=["Effect estimation?"])
        .set_properties(subset=["Title"], **{
            "min-width": "430px",
            "max-width": "430px",
            "text-align": "left",
            "white-space": "normal"
        })
        .set_properties(subset=["Journal"], **{
            "min-width": "170px",
            "max-width": "170px",
            "text-align": "left",
            "white-space": "normal"
        })
        .set_properties(subset=["Reason"], **{
            "min-width": "360px",
            "max-width": "360px",
            "text-align": "left",
            "white-space": "normal"
        })
        .set_properties(**{
            "font-size": "12px",
            "text-align": "center",
            "vertical-align": "top"
        })
    )

## 13. Display the final ranked shortlist

In [51]:
build_final_view(df_final, n=TOP_N_FOR_LLM)

,Rank,Title,Year,Journal,Heuristic,Final,Gemini label,Effect estimation?,Action,Group,QuantSignal,Reason,URL
0,1,How is participation in parent-child-interaction-focused andparenting-skills-focused courses associated with childdevelopment?,2017,Early Years Journal ofInternational Researchand Development,0.647,2.565,highly_relevant,True,2,2,2,Directly examines effects of a family support programon children's socio-emotional development withquantitative analysis.,https://doi.org/10.1080/09575146.2017.1288089
1,2,Stimulating Parenting Practices in Indigenous andNon-Indigenous Mexican Communities,2017,International Journal ofEnvironmental Researchand Public Health,0.597,2.560,highly_relevant,True,2,2,2,"Examines associations between factors and stimulatingparenting practices in Mexican communities, containingquantitative data relevant for understanding parenting.",https://doi.org/10.3390/ijerph15010029
2,3,Nurturing Sustainable Development: The Interplay ofParenting Styles and SDGs in Children’s Development,2024,Children,0.574,2.557,highly_relevant,True,2,2,2,"Investigates the impact of parenting styles onchildren's cognitive and social-affective developmentusing empirical data, highly relevant to familysupport.",https://doi.org/10.3390/children11060695
3,4,Collateral benefits of the family check-up on earlychildhood school readiness: Indirect effects of parents'positive behavior support.,2008,Developmental Psychology,0.544,2.554,highly_relevant,True,2,2,2,Examined longitudinal effects of a family intervention(FCU) on early childhood school readiness.,https://doi.org/10.1037/a0013858
4,5,A Longitudinal Study of the Relationship of MaternalAutonomy Support to Children's Adjustment and Achievement inSchool,2005,Journal of Personality,0.539,2.554,highly_relevant,True,2,2,2,Longitudinal study examining relations of maternalautonomy support to children's school and socialadjustment.,https://doi.org/10.1111/j.1467-6494.2005.00347.x
5,6,"Health Care, Family, and Community Factors Associated withMental, Behavioral, and Developmental Disorders in EarlyChildhood — United States, 2011–2012",2016,MMWR Morbidity andMortality Weekly Report,0.596,2.460,highly_relevant,True,1,2,2,"Identifies associations between family/communityfactors and developmental disorders in early childhood,containing quantitative evidence of risk factors.",https://doi.org/10.15585/mmwr.mm6509a1
6,7,Family interaction and a supportive social network assalutogenic factors in childhood atopic illness,2002,Pediatric Allergy andImmunology,0.538,2.104,somewhat_relevant,True,2,2,2,Prospective study of family interaction and socialnetwork on allergy development in infants; outcome notsocio-emotional.,https://doi.org/10.1034/j.1399-3038.2002.00086.x
7,8,"The Relationships Among Adaptive Behaviors of Children withAutism, Family Support, Parenting Stress, and Coping",2011,Issues in ComprehensivePediatric Nursing,0.548,2.025,somewhat_relevant,True,2,1,2,"Examines relationships among family support, parentingstress, and adaptive behaviors in children with autism.",https://doi.org/10.3109/01460862.2011.555270
8,9,Parenting stress in mothers of children with an intellectualdisability: the effects of parental cognitions in relationto child characteristics and family support,2005,Journal of IntellectualDisability Research,0.542,2.024,somewhat_relevant,True,2,1,2,Examines effects of parental cognitions and familysupport on parenting stress for mothers of childrenwith intellectual disability.,https://doi.org/10.1111/j.1365-2788.2005.00673.x
9,10,Emotional Warmth and Rejection Parenting Styles ofGrandparents/Great Grandparents and the Social–EmotionalDevelopment of Grandchildren/Great Grandchildren,2023,International Journal ofEnvironmental Researchand Public Health,0.622,2.012,somewhat_relevant,True,1,2,2,"Examines associations between parenting styles andchild socio-emotional development, providingquantitative evidence (observational).",https://doi.org/10.3390/ijerph20021568
